In [1]:
# Importing libraries
import pandas as pd
import numpy as np
import os

In [8]:
# Load matrices
# Choose the appropriate directory and files
directory = "/home/helitonmrf/OneDrive/Documentos/QSAR/Tetronamides/QSAR 2D/QSARModeling_outputs"
X_matrix_file = 'Assay1_correlationcut_output.csv'
y_matrix_file = 'Bioactivity1.txt'

df = pd.read_csv(os.path.join(directory,X_matrix_file),sep=';',index_col=0)
X = df.to_numpy()
y = pd.read_csv(os.path.join(directory,y_matrix_file),sep=';',header=None).values

array([], shape=(33, 0), dtype=float64)

In [3]:
# Run Cross-validation
from cross_validation_class import CrossValidation
cv = CrossValidation(X,y)
print(cv.Q2())

ValueError: Found array with 0 feature(s) (shape=(32, 0)) while a minimum of 1 is required by the scale function.

In [24]:
# Plot R² and Q² for different number of latent variables
from bokeh.plotting import figure, output_notebook, show
from bokeh.models import LabelSet, Label, ColumnDataSource

output_notebook()

source = ColumnDataSource(data=dict(
    lv=range(1,12),
    R2=cv.R2(),
    Q2=cv.Q2(),
))

TOOLTIPS = [
    ("LV", "@lv"),
    ("R2", "@R2"),
    ("Q2", "@Q2")
]

p = figure(title="Cross-validation error - PLS", x_axis_label='Latent variables number', y_axis_label='Q² or R²',
          tooltips = TOOLTIPS)
    #p.text(x+0.3,y+0.3,flav.index[i])
p.line("lv","R2",legend="R²", source=source)
p.circle("lv","R2",legend="R²", source=source)
p.line("lv","Q2",color="red", legend="Q²", source=source)
p.circle("lv","Q2",color="red", legend="Q²", source=source)
p.legend.location = "top_left"
show(p)

Loading BokehJS ...

In [9]:
# If you want the number of latent variables be chosen automatically run this cell
nLV = np.argmax(cv.Q2())+1
nLV

5

In [ ]:
# If you want to choose the number of latent variables manually type the desired number of LV
nLV = 6

In [10]:
# See a summary of the cross-validation results
cv.returnParameters(nLV)

,0
PRESS,4.269892
R2,0.681640
RMSEC,0.413274
rcal,0.825615
avgRmcal,0.573148
deltaRmcal,0.216984
PRESSCV,5.880052
Q2,0.561587
RMSECV,0.484976
rcv,0.756202


In [11]:
# Plot experimental X predited values of y for calibration
from bokeh.plotting import figure, output_notebook, show
from bokeh.models import LabelSet, Label, ColumnDataSource

output_notebook()

source = ColumnDataSource(data=dict(
    y=y,
    y_pred=cv.ycal[:,nLV-1],
))

TOOLTIPS = [
    ("y", "@y"),
    ("y_pred", "@y_pred")
]

p = figure(title="Calibration prediction", x_axis_label='Experimental pIC50', y_axis_label='Predicted pIC50',
          tooltips = TOOLTIPS)
    #p.text(x+0.3,y+0.3,flav.index[i])
p.circle("y","y_pred", source=source)
p.line(y[:,0],y[:,0],color="red")
show(p)

Loading BokehJS ...

In [12]:
# Plot experimental X predited values of y for cross-validation
from bokeh.plotting import figure, output_notebook, show
from bokeh.models import LabelSet, Label, ColumnDataSource

output_notebook()

source = ColumnDataSource(data=dict(
    y=y,
    y_pred=cv.ycv[:,nLV-1],
))

TOOLTIPS = [
    ("y", "@y"),
    ("y_pred", "@y_pred")
]

p = figure(title="Cross-validation prediction", x_axis_label='Experimental pIC50', y_axis_label='Predicted pIC50',
          tooltips = TOOLTIPS)
    #p.text(x+0.3,y+0.3,flav.index[i])
p.circle("y","y_pred", source=source)
p.line(y[:,0],y[:,0],color="red")
show(p)

Loading BokehJS ...

In [ ]:
# Save cross validation results file
output_cv_file = "saidacv.csv"
cv.saveParameters(directory+output_cv_file)

In [13]:
# y-randomization
from yrandomization import YRandomization
# Here choose the number of randomizations you want
n_randomizations = 50
yr = YRandomization(X,y,nLV,n_randomizations)

In [14]:
# Plot R² and Q² values obtained in y-randomization
from bokeh.plotting import figure, output_notebook, show
from bokeh.models import LabelSet, Label, ColumnDataSource

output_notebook()

source = ColumnDataSource(data=dict(
    R2=yr.R2,
    Q2=yr.Q2,
))

TOOLTIPS = [
    ("R2", "@R2"),
    ("Q2", "@Q2")
]

p = figure(title="y-randomization", x_axis_label='R²', y_axis_label='Q²',
          tooltips = TOOLTIPS)
    #p.text(x+0.3,y+0.3,flav.index[i])
p.circle("R2","Q2", source=source)
show(p)

Loading BokehJS ...

In [15]:
# Plot Correlation between y randomized values and real y against R² and Q² according to 
# Eriksson, L., Jaworska, J., Worth, A. P., Cronin, M. T. D., McDowell, R. M., & Gramatica, P. (2003). 
# Methods for reliability and uncertainty assessment and for applicability evaluations of classification- and 
# regression-based QSARs. Environmental Health Perspectives, 111(10), 1361–1375. https://doi.org/10.1289/ehp.5758
from bokeh.plotting import figure, output_notebook, show
from bokeh.models import LabelSet, Label, ColumnDataSource
from bokeh.layouts import gridplot

output_notebook()

source1 = ColumnDataSource(data=dict(
    R2=yr.R2,
    R=yr.R,
))

aR2,bR2 = yr.returnRegResultsR2()

xR2 = np.linspace(np.min(yr.R),np.max(yr.R))
yR2 = aR2[0]*xR2 + bR2

TOOLTIPS = [
    ("R", "@R"),
    ("R2", "@R2")
]

s1 = figure(title="y-randomization", x_axis_label='R', y_axis_label='R²',
          tooltips = TOOLTIPS)
    #p.text(x+0.3,y+0.3,flav.index[i])
s1.circle("R","R2", source=source1)
s1.line(xR2, yR2,color="red",legend="{:.2f}x + {:.2f}".format(aR2[0][0],bR2[0]))
s1.legend.location = "top_left"

source2 = ColumnDataSource(data=dict(
    Q2=yr.Q2,
    R=yr.R,
))

aQ2,bQ2 = yr.returnRegResultsQ2()

xQ2 = np.linspace(np.min(yr.R),np.max(yr.R))
yQ2 = aQ2[0]*xQ2 + bQ2

TOOLTIPS = [
    ("R", "@R"),
    ("Q2", "@Q2")
]

s2 = figure(title="y-randomization", x_axis_label='R', y_axis_label='Q²',
          tooltips = TOOLTIPS)
    #p.text(x+0.3,y+0.3,flav.index[i])
s2.circle("R","Q2", source=source2)
s2.line(xQ2, yQ2,color="red",legend="{:.2f}x + {:.2f}".format(aQ2[0][0],bQ2[0]))
s2.legend.location = "top_left"

p = gridplot([[s1, s2]])

show(p)

Loading BokehJS ...

In [16]:
# Saving y-randomization results

output_yr_file = "yrandomization.csv"

yrMatrix = [yr.R2,yr.Q2,yr.R]
dfyr = pd.DataFrame(columns = ["R2","Q2","R(yrd,y)"],data=np.transpose(yrMatrix))
dfyr.to_csv(directory+output_yr_file, sep =',', index=False)

In [21]:
# Leave-N-out
from lno import LNO

#Here you can choose the number of repetitions
n_repetitions = 10

lno = LNO(X,y,nLV,nrepet=n_repetitions)
dfLNO = pd.DataFrame(data=lno.Q2)

In [22]:
# Plot the graph with leave-N-out results
from bokeh.models import ColumnDataSource, Whisker, Range1d
from bokeh.plotting import figure, show, output_notebook
from bokeh.sampledata.autompg import autompg as df

output_notebook()

p = figure(title="Leave-N-Out",x_axis_label='N', y_axis_label='Q²',)

lno_mean = dfLNO.mean(1)
lno_std = dfLNO.std(1)

base = range(1,dfLNO.shape[0]+1)
upper = lno_mean + lno_std
lower = lno_mean - lno_std
    
source_error = ColumnDataSource(data=dict(base=base, lower=lower, upper=upper))

p.add_layout(
    Whisker(source=source_error, base="base", upper="upper", lower="lower")
)

p.circle(x=base,y=lno_mean,color="black")

p.x_range = Range1d(0, len(base)+1)
p.y_range = Range1d(np.min(lower)-0.1,np.max(lower)+0.1)

show(p)

Loading BokehJS ...

In [23]:
# Saving Leave-N-out results
dfLNO.index = ["Leave-{}-out".format(i+1) for i in range(len(dfLNO))]
dfLNO.columns = ["Repetition {}".format(i+1) for i in range(dfLNO.shape[1])]
dfLNO.to_csv(directory+"LNO.csv")